# Capítulo 2: Git y Control de Versiones — Laboratorio práctico
**Tópicos Avanzados para Data Science en Python**

En este notebook practicaremos el flujo completo de Git en Google Colab:

| Tema | Comandos |
|---|---|
| Configuración | `git --version`, `git config` |
| Repositorios | `git clone`, `git remote` |
| Flujo básico | `git status`, `git add`, `git commit` |
| Archivos ignorados | `.gitignore`, `git check-ignore`, `git rm --cached` |
| Sincronización | `git push`, `git pull`, `git fetch` |
| Historial | `git log` |
| Ramas | `git branch`, `git switch`, `git merge` |
| Colaboración | forks, `upstream`, conflictos |

### Las 4 zonas de Git
```
 Working Directory  --git add-->  Staging Area  --git commit-->  Local Repository  --git push-->  Remote (GitHub)
        ^                                                                                              |
        +------------------------------------------- git pull ----------------------------------------+
```

### ¿Cómo practicamos *push*, *pull* y *forks* sin cuenta de GitHub?
Git puede usar como **remoto** cualquier repositorio, incluso una carpeta de la misma máquina.
Crearemos una carpeta `github/` que hará el papel de GitHub, y dos carpetas `usuario1/` y `usuario2/` que simulan las computadoras de dos personas del equipo:

```
/content/git_lab/
├── github/        ← "GitHub" simulado (repositorios remotos)
├── usuario1/      ← computadora del usuario 1
└── usuario2/      ← computadora del usuario 2
```

Los comandos son **exactamente los mismos** que con GitHub; solo cambia la URL (`/content/git_lab/github/demo.git` en lugar de `https://github.com/usuario/demo.git`).

> 💡 En Colab no hay un editor interactivo para Git: haz los commits siempre con `-m "mensaje"` y los merges con `--no-edit`.

## 1. Configuración inicial
Git necesita saber **quién eres** para firmar cada commit.

| Comando | ¿Qué hace? |
|---|---|
| `git --version` | Verifica la instalación |
| `git config --global user.name "..."` | Nombre del autor de los commits |
| `git config --global user.email "..."` | Correo (el mismo de tu cuenta de GitHub) |
| `git config --global core.autocrlf input` | Saltos de línea en Linux/Mac/Colab (`true` en Windows) |
| `git config --global --list` | Muestra la configuración |

`--global` aplica a todos los repositorios del usuario; sin `--global` aplica solo al repositorio actual.

In [ ]:
!git --version

In [ ]:
!git config --global user.name "Estudiante BSG"
!git config --global user.email "estudiante@bsg.edu"
!git config --global core.autocrlf input
!git config --global init.defaultBranch main
# git pull = fetch + merge (comportamiento clásico, el que vemos en la PPT)
!git config --global pull.rebase false
!git config --global --list

In [ ]:
# La configuración global vive en un archivo oculto del home (¡recuerda el Capítulo 1!)
!cat ~/.gitconfig

### ✏️ Ejercicio 1.1 — Tu identidad
1. Configura **tu** nombre y **tu** correo de forma global.
2. Muestra solo el valor de `user.email` (pista: `git config user.email`).
3. Lista toda la configuración global y verifica que `core.autocrlf` sea `input`.
4. Responde en un comentario: ¿por qué en Colab usamos `input` y no `true`?

In [ ]:
# Tu código aquí

## 2. Preparando el laboratorio
Ejecuta esta celda para crear el "GitHub" simulado con tres repositorios remotos:

| Repositorio remoto | Uso |
|---|---|
| `github/demo.git` | Ejemplos de la teoría |
| `github/proyecto_ventas.git` | Proyecto de tus ejercicios |
| `github/bsg/plantilla_proyecto.git` | Repositorio "ajeno" (de BSG) para practicar forks |

> Si en algún momento quieres empezar de cero, vuelve a ejecutar esta celda.

In [ ]:
%%bash
rm -rf /content/git_lab
mkdir -p /content/git_lab/{github/bsg,usuario1,usuario2}

# Crea un repositorio remoto "vacío" (bare) y le sube un commit inicial con los archivos de $2
crear_remoto () {
  git init -q --bare -b main "$1"
  tmp=$(mktemp -d)
  git -C "$tmp" init -q -b main
  cp -r "$2"/. "$tmp"/
  git -C "$tmp" add .
  git -C "$tmp" -c user.name="GitHub" -c user.email="noreply@github.com" commit -q -m "Initial commit"
  git -C "$tmp" push -q "$1" main
  rm -rf "$tmp"
}

src=$(mktemp -d)

# --- demo.git ---
mkdir -p $src/demo && echo "# demo" > $src/demo/README.md
crear_remoto /content/git_lab/github/demo.git $src/demo

# --- proyecto_ventas.git ---
mkdir -p $src/ventas
cat > $src/ventas/README.md <<'EOF'
# Proyecto Ventas
Estado: en desarrollo
EOF
cat > $src/ventas/ventas.csv <<'EOF'
id,fecha,producto,monto
1,2025-01-05,Laptop,3700
2,2025-01-19,Tablet,1500
3,2025-02-14,Mouse,40
EOF
crear_remoto /content/git_lab/github/proyecto_ventas.git $src/ventas

# --- bsg/plantilla_proyecto.git (repositorio "ajeno") ---
mkdir -p $src/plantilla
cat > $src/plantilla/README.md <<'EOF'
# Plantilla de proyecto de Data Science (BSG)
Estructura sugerida: data/, notebooks/, src/
EOF
echo "pandas" > $src/plantilla/requirements.txt
crear_remoto /content/git_lab/github/bsg/plantilla_proyecto.git $src/plantilla

rm -rf $src
echo "Laboratorio listo:"
find /content/git_lab -maxdepth 3 -name "*.git"

## 3. `git clone` — Traer un repositorio a tu máquina
`git clone <url>` descarga el repositorio **con toda su historia** y deja configurado el remoto `origin`.

En GitHub la URL sale del botón **Code → HTTPS**; aquí usamos la ruta de nuestro "GitHub" simulado.

In [ ]:
%cd /content/git_lab/usuario1
!git clone /content/git_lab/github/demo.git

In [ ]:
%cd /content/git_lab/usuario1/demo
# La carpeta oculta .git ES el repositorio (historia, ramas, configuración)
!ls -la

In [ ]:
# ¿A dónde apunta origin?
!git remote -v
!git log --oneline

### ✏️ Ejercicio 3.1 — Clonar el proyecto
1. Clona `/content/git_lab/github/proyecto_ventas.git` dentro de `/content/git_lab/usuario1`.
2. Entra a la carpeta clonada y lista **todo** su contenido (incluidos ocultos). ¿Qué carpeta oculta aparece?
3. Muestra a dónde apunta el remoto `origin`.
4. ¿Cuántos commits tiene el proyecto y quién es su autor? (pista: `git log`)

In [ ]:
# Tu código aquí

## 4. `git status` y `git add`
- `git status`: muestra en qué zona está cada archivo. **Úsalo antes y después de cada paso.**
- `git add`: mueve cambios del *working directory* al *staging area*.

| Comando | ¿Qué hace? |
|---|---|
| `git add archivo` | Agrega un archivo |
| `git add *.txt` | Agrega por patrón |
| `git add .` | Agrega todo lo que cambió |
| `git restore --staged archivo` | Saca un archivo del staging |
| `git status --short` | Versión compacta: `??` sin rastrear, `A` agregado, `M` modificado |

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git status

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
echo "Hola mundo" > prueba.txt
echo "Hola mundo 2" > prueba_2.txt
git status      # ambos aparecen como "Untracked files"

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git add prueba.txt
git status      # prueba.txt en staging (Changes to be committed), prueba_2.txt sin rastrear

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git add *.txt
git status --short
echo "--- saco prueba_2.txt del staging ---"
git restore --staged prueba_2.txt
git status --short

### ✏️ Ejercicio 4.1 — Preparar cambios
Trabaja en `/content/git_lab/usuario1/proyecto_ventas`:
1. Crea `limpieza.py` con el contenido `import pandas as pd`, y los archivos `notas.txt` y `borrador.txt` con cualquier texto.
2. Revisa el estado del repositorio.
3. Agrega **solo** `limpieza.py` al staging y revisa con `git status --short`.
4. Agrega todos los `.txt` usando un patrón.
5. Te arrepentiste: saca `borrador.txt` del staging.
6. Explica en un comentario qué significan `A` y `??` en el resultado final.

In [ ]:
%%bash
# Tu código aquí

## 5. `git commit` — Guardar una versión
`git commit -m "mensaje"` guarda **lo que está en staging** como una nueva versión en el repositorio local.

**Buenos mensajes:** verbo + qué cambió → `"Agregar limpieza de nulos en ventas"`.
**Malos mensajes:** `"cambios"`, `"fix"`, `"asdf"`.

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git add prueba.txt prueba_2.txt
git commit -m "Agregar archivos de prueba"
git status      # "ahead of 'origin/main' by 1 commit" -> el commit solo existe en tu máquina

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
echo "Nuevo ingreso" >> prueba_2.txt
git status --short    # M = modificado
git diff              # ¿qué cambió exactamente?

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
# -a agrega automáticamente los archivos YA rastreados que cambiaron
git commit -am "Actualizar prueba_2 con nuevo ingreso"
git log --oneline

### ✏️ Ejercicio 5.1 — Tus primeros commits
En `proyecto_ventas`:
1. Haz commit de lo que está en staging (del ejercicio anterior) con un mensaje descriptivo.
2. Verifica con `git status` cuántos commits estás adelantado respecto a `origin/main`.
3. Agrega una nueva venta al final de `ventas.csv`: `4,2025-03-02,Monitor,850`.
4. Muestra la diferencia con `git diff` y luego haz commit **sin usar** `git add` por separado.
5. Muestra el historial en una línea por commit.

In [ ]:
%%bash
# Tu código aquí

## 6. `.gitignore` — Lo que nunca debe llegar a GitHub
Archivo de texto en la raíz del repo; cada línea es un patrón a ignorar:

```
.env              # credenciales
NotConsidered/    # una carpeta completa
*.csv             # por extensión
__pycache__/
.ipynb_checkpoints/
```

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
echo "password=123456" > .env
mkdir -p NotConsidered && echo "idea a medias" > NotConsidered/borrador.txt
git status --short     # .env y NotConsidered/ aparecen: ¡peligro con git add .!

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
printf ".env\nNotConsidered/\n" > .gitignore
cat .gitignore
echo "--- status ---"
git status --short     # ya solo aparece .gitignore
echo "--- ¿qué regla ignora .env? ---"
git check-ignore -v .env

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git add .
git commit -m "Agregar .gitignore"

### ✏️ Ejercicio 6.1 — Proteger el proyecto
En `proyecto_ventas`:
1. Crea estos archivos que **no** deben subirse:
   - `.env` con `DB_PASSWORD=super_secreta`
   - `data/raw/ventas_completas.csv` (cualquier contenido)
   - `__pycache__/limpieza.cpython-310.pyc`
   - `modelo.pkl`
2. Crea un `.gitignore` que los ignore a todos (usa una regla por carpeta y una por extensión `.pkl`).
3. Comprueba con `git status --short` que solo aparecen `.gitignore` y `borrador.txt`.
4. Usa `git check-ignore -v` para ver qué regla ignora `modelo.pkl`.
5. Haz commit **solo** del `.gitignore`.

In [ ]:
%%bash
# Tu código aquí

### ✏️ Ejercicio 6.2 — ¡Ups, subí una credencial!
`.gitignore` **no** saca archivos que ya fueron commiteados.
1. Crea `credenciales.txt` con `token=abc123`, agrégalo y haz commit (el error).
2. Agrega `credenciales.txt` al `.gitignore`.
3. Quita el archivo del control de versiones **sin borrarlo de tu disco** (pista: `git rm --cached`).
4. Haz commit del arreglo y verifica que el archivo sigue existiendo con `ls`.
5. Comenta: ¿el token sigue en el historial? ¿Qué deberías hacer con él en la vida real?

In [ ]:
%%bash
# Tu código aquí

## 7. `git push` y `git log`
- `git push`: sube tus commits locales al remoto (`origin`).
- `git log`: muestra el historial.

| Comando | ¿Qué hace? |
|---|---|
| `git push` | Sube la rama actual |
| `git push --set-upstream origin rama` | Sube una rama nueva y la asocia (abreviado `-u`) |
| `git log --oneline` | Un commit por línea |
| `git log --oneline --graph --all` | Dibuja todas las ramas |
| `git log -n 3` | Últimos 3 commits |

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git push
git status

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git log --oneline       # HEAD -> main (tú) y origin/main (el remoto) están en el mismo commit
echo "--- lo que tiene 'GitHub' ---"
git --git-dir=/content/git_lab/github/demo.git log --oneline

### ✏️ Ejercicio 7.1 — Publicar el trabajo
En `proyecto_ventas`:
1. Sube tus commits al remoto.
2. Verifica que el "GitHub" simulado (`/content/git_lab/github/proyecto_ventas.git`) tiene tus commits.
3. Muestra solo los **2 últimos** commits.
4. Cuenta cuántos commits tiene la rama (pista: `git rev-list --count HEAD`).
5. Muestra el historial con el formato `hash autor mensaje` (pista: `--format='%h %an %s'`).

In [ ]:
%%bash
# Tu código aquí

## 8. Ramas: `git branch`, `git switch` y `git merge`

| Comando | ¿Qué hace? |
|---|---|
| `git branch` | Lista ramas (`*` = rama actual) |
| `git branch nueva` | Crea una rama (no te mueve) |
| `git switch nueva` | Cambia de rama (antes: `git checkout`) |
| `git switch -c nueva` | Crea y cambia en un paso |
| `git merge rama` | Trae los cambios de `rama` a la rama actual |
| `git branch -d rama` | Borra una rama ya fusionada |

**Regla del merge:** te paras en la rama **destino** y mergeas la rama **origen**.

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git branch new-branch
git branch
git switch new-branch
echo "Modificación" >> prueba.txt
git commit -am "Modificar prueba.txt en new-branch"
echo "--- prueba.txt en new-branch ---"
cat prueba.txt

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git switch main
echo "--- prueba.txt en main (aún sin el cambio) ---"
cat prueba.txt
git merge new-branch          # Fast-forward
echo "--- prueba.txt en main después del merge ---"
cat prueba.txt

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
# Rama nueva publicada en el remoto
git switch -c dev
echo "test 3" > test3.txt
git add test3.txt
git commit -m "Agregar test3.txt en dev"
git push --set-upstream origin dev
echo "--- ramas locales y remotas ---"
git branch -a
git log --oneline --graph --all

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
git switch main
git push
git branch -d new-branch      # ya fue fusionada
git branch

### ✏️ Ejercicio 8.1 — Trabajar en una rama
En `proyecto_ventas`:
1. Crea y cámbiate en un solo paso a la rama `feature/limpieza`.
2. Agrega al final de `limpieza.py` la línea `df = pd.read_csv("ventas.csv").dropna()` y haz commit.
3. Vuelve a `main` y comprueba con `cat` que el cambio **no** está.
4. Fusiona `feature/limpieza` en `main` y comprueba que ahora sí está. ¿Qué tipo de merge fue?
5. Sube `main` al remoto y borra la rama `feature/limpieza`.

In [ ]:
%%bash
# Tu código aquí

### ✏️ Ejercicio 8.2 — Ramas por experimento (ds1 / ds2)
Simula el flujo de la PPT donde cada científico de datos trabaja en su rama:
1. Desde `main`, crea la rama `ds1` con un archivo `modelo_rf.py` (`# Random Forest`) y haz commit.
2. Vuelve a `main`, crea la rama `ds2` con `modelo_xgb.py` (`# XGBoost`) y haz commit.
3. Publica **ambas** ramas en el remoto asociándolas (`--set-upstream`).
4. Muestra todas las ramas (locales y remotas) y el gráfico del historial.
5. El experimento ganador fue `ds2`: fusiónalo en `main` y súbelo.

In [ ]:
%%bash
# Tu código aquí

## 9. `git pull` — Trabajar en equipo
`git pull origin main` = `git fetch` (descargar) + `git merge` (combinar).

Simularemos a un **segundo usuario** que clona el mismo repositorio en otra "computadora".

In [ ]:
%%bash
cd /content/git_lab/usuario2
git clone -q /content/git_lab/github/demo.git
cd demo
# Identidad SOLO para este repositorio (sin --global): es "otra persona"
git config user.name "Usuario 2"
git config user.email "usuario2@bsg.edu"
echo "Aporte del usuario 2" > aporte_u2.txt
git add aporte_u2.txt
git commit -m "Agregar aporte del usuario 2"
git push

In [ ]:
%%bash
cd /content/git_lab/usuario1/demo
echo "--- antes del pull ---"
ls
git pull origin main
echo "--- después del pull ---"
ls
git log --format='%h %an: %s' -3

### ✏️ Ejercicio 9.1 — Colaboración
1. Como **usuario 2**: clona `proyecto_ventas` en `/content/git_lab/usuario2`, configura una identidad local (`Usuario 2`), crea `reporte.md` con el texto `# Reporte mensual`, haz commit y push.
2. Como **usuario 1**: comprueba que aún no tienes `reporte.md`.
3. Usa `git fetch` y luego `git log --oneline main..origin/main` para ver qué commits llegaron **antes** de combinarlos.
4. Actualiza tu repositorio con `git pull` y muestra los últimos 3 commits con su autor.

In [ ]:
%%bash
# Tu código aquí

## 10. Forks
Un **fork** es una copia de un repositorio **ajeno** en **tu** cuenta de GitHub. En GitHub se crea con el botón **Fork**; aquí lo simulamos con `git clone --bare` (una copia del repositorio dentro de "GitHub").

Flujo típico: **fork → clone → cambios → push a tu fork → (pull request al original)**.
Para mantener tu fork al día se agrega el remoto `upstream` (el original).

In [ ]:
%%bash
cd /content/git_lab/github
mkdir -p usuario1
# "Botón Fork": copia de bsg/plantilla_proyecto en la cuenta de usuario1
git clone -q --bare bsg/plantilla_proyecto.git usuario1/plantilla_demo_fork.git
ls bsg usuario1

In [ ]:
%%bash
cd /content/git_lab/usuario1
git clone -q /content/git_lab/github/usuario1/plantilla_demo_fork.git
cd plantilla_demo_fork
echo "numpy" >> requirements.txt
git commit -am "Agregar numpy a requirements"
git push
echo "--- mi fork ---"
git log --oneline
echo "--- el original de BSG (no cambió) ---"
git --git-dir=/content/git_lab/github/bsg/plantilla_proyecto.git log --oneline

In [ ]:
%%bash
cd /content/git_lab/usuario1/plantilla_demo_fork
git remote add upstream /content/git_lab/github/bsg/plantilla_proyecto.git
git remote -v      # origin = mi fork, upstream = el original

### ✏️ Ejercicio 10.1 — Fork y sincronización
1. Haz un "fork" de `github/bsg/plantilla_proyecto.git` con el nombre `github/usuario1/plantilla_proyecto.git`.
2. Clona tu fork en `/content/git_lab/usuario1/plantilla_proyecto`, crea `analisis.py` (`# Análisis exploratorio`), haz commit y push.
3. Comprueba que el repositorio original de BSG **no** tiene tu commit.
4. Agrega el remoto `upstream` apuntando al original.
5. Ejecuta la **celda auxiliar** de abajo (BSG publica una actualización en el original).
6. Trae esa actualización a tu fork: `git fetch upstream`, `git merge --no-edit upstream/main` y `git push`.
7. Muestra el gráfico del historial: ¿qué commit nuevo creó el merge?

In [ ]:
%%bash
# Celda auxiliar: BSG actualiza su repositorio original
tmp=$(mktemp -d)
git clone -q /content/git_lab/github/bsg/plantilla_proyecto.git $tmp
cd $tmp
echo "scikit-learn" >> requirements.txt
git -c user.name="BSG Institute" -c user.email="bsg@bsg.edu" commit -qam "Agregar scikit-learn"
git push -q
rm -rf $tmp
echo "BSG publicó un nuevo commit en el original"

In [ ]:
%%bash
# Tu código aquí

## 11. 🏆 Reto integrador: resolver un conflicto
Un **conflicto** ocurre cuando dos personas cambian **la misma línea** de un archivo. Git no sabe cuál versión dejar y marca el archivo así:

```
<<<<<<< HEAD
tu versión
=======
la versión del otro
>>>>>>> 1a2b3c4
```

Resolverlo = editar el archivo dejando el contenido final (sin las marcas) → `git add` → `git commit`.

En `proyecto_ventas`:
1. Ambos usuarios hacen `git pull` para partir actualizados.
2. **Usuario 2** cambia la línea `Estado: en desarrollo` del `README.md` por `Estado: en revisión`, hace commit y push.
3. **Usuario 1** (sin hacer pull) cambia la misma línea por `Estado: listo para producción`, hace commit e intenta hacer push. ¿Qué pasa?
4. Usuario 1 hace `git pull`: observa el conflicto con `git status` y `cat README.md`.
5. Resuelve el conflicto dejando `Estado: listo para producción (revisado)`, termina el merge y sube los cambios.
6. Usuario 2 hace `git pull` y ambos muestran `git log --oneline --graph` para comprobar que tienen la misma historia.

> Pista: para reemplazar una línea desde la terminal puedes usar `sed -i 's/texto viejo/texto nuevo/' archivo` (Capítulo 1).

In [ ]:
%%bash
# Tu código aquí

## 12. (Opcional) Conectar Colab con tu GitHub real
Para trabajar con un repositorio real necesitas un **Personal Access Token** (PPT, diapositiva 25).

⚠️ **Nunca escribas el token en el notebook.** Guárdalo en el panel **Secretos** de Colab (ícono de llave 🔑 a la izquierda) con el nombre `GITHUB_TOKEN` y léelo con `userdata`:

In [ ]:
from google.colab import userdata

usuario = "tu_usuario_github"          # <- cambia esto
repo = "tu_repositorio"                # <- cambia esto
token = userdata.get("GITHUB_TOKEN")

%cd /content
!git clone https://{token}@github.com/{usuario}/{repo}.git
%cd {repo}
# Quitamos el token de .git/config para que no quede guardado en texto plano
!git remote set-url origin https://github.com/{usuario}/{repo}.git
!git remote -v

In [ ]:
# Para subir cambios, pasa el token solo en el momento del push
!git push https://{token}@github.com/{usuario}/{repo}.git main

---
## 🎯 Fin de los ejercicios
Las soluciones están en el notebook **`Capitulo2_Soluciones.ipynb`**. Intenta resolver todo antes de revisarlas.